<a href="https://colab.research.google.com/github/nika19du/Math-Music-Project/blob/main/Math_Music_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Mathematics behind music: notes, intervals and chords

**Math concepts for developers - course project (work in progress)

> this notebook is an evloving investigation. The experiments and conclusions will be expanded as the project develops.

## 1. Introduction
Music can be described using mathematical structures. The twelve pitch classes is Western equal temperament form a repeating cycle, intervals can be represented as differences and chords can be constructed using fixed semitone patterns.

** Research question **: How can numerical and vector representations help us construct analyze and transform melodies and chords?

**Objectives**:
1. Represent notes and pitch classes numerically.
2. Calculate melodic intervals using vector and matrix operations.
3. Construct major and minor triads from interval patterns.
4. Validate the mathematical models with Python experiments

**Scope and assumptions:** we use twelve-tone qual-tempered system (12-TET). We focus on pitch and ordered note sequences, not rhythm, dynamics, timbre or full harmonic analysis. Pitch-class values discard octave informtation. In this notebook, note names use sharps for simplicity, enharmonic spelling depend on musical context.

## 2. Mathematical Foundations: Semitones and Pitch Classes

In 12-TET, an ocatave contains **12 equal semitone steps**. We encode the pitch classes using integers from $0$ to $11$:

| Note | C | C♯ | D | D♯ | E | F | F♯ | G | G♯ | A | A♯ | B |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Pitch class | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |

Becuase the pitch classes repeat every octave, we use **modular arithmetic**:
$$ p = n \bmod 12. $$

For example, $12\bmod 12=0 $ and $14\bmod 12=2$. Thus pitch classes $0$ and $12$ both represent C, although the latter is one octave higher in our semitone coordinate system.

**Important distinction:** A pitch class (modulo 12) is not the same as a pitch height. For computing directed melodic intervals across octaves, we retain integer semitone heights without reducing them modulo 12.

In 12-TET, the frequency ration of two pitches separated by $k$ semitones is:
$$\frac{f_2}{f_1}=2^{k/12}.$$

For $k=12$ this becomes $f_2=2f_1$ the octave relationship.


In [1]:
 NOTE_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

def pitch_class(semitone_height: int) -> int:
  """
  Return the pitch class(0..11) of an integer semitone height.
  """
  return semitone_height  % 12

def note_name(semitone_height: int) -> str:
  """
  Return a simplified sharp-based note name (without octave).
  """
  return NOTE_NAMES[pitch_class(semitone_height)]

for height in [0, 4, 7, 9, 12, 14]:
    print(f"height={height:2d}  pitch_class={pitch_class(height):2d}  note={note_name(height)}")

height= 0  pitch_class= 0  note=C
height= 4  pitch_class= 4  note=E
height= 7  pitch_class= 7  note=G
height= 9  pitch_class= 9  note=A
height=12  pitch_class= 0  note=C
height=14  pitch_class= 2  note=D


## 3. Melodies as Vectors and Intervals as Differences

Consider four consecutive note heights represented by the vector

$$\mathbf{m}=\begin{bmatrix}0\\4\\7\\9\end{bmatrix}.$$

Their **successive directed intervals** are $4-0=4$, $7-4=3$ and $9-7=2$ semiteones. We can compute these intervals using the matrix

$$D=\begin{bmatrix}-1&1&0&0\\0&-1&1&0\\0&0&-1&1\end{bmatrix}.$$

Then

$$D\mathbf{m}=\begin{bmatrix}4\\3\\2\end{bmatrix}.$$

This matrix represents a linear transformation
$D:\mathbb{R}^{4}\rightarrow\mathbb{R}^{3}$.
It maps four note heights to three adjacent differences. In general, a sequence of $n$ notes produces $n-1$ successive intervals.

**Why the starting note matters.** Two melodiec can have the same intervals:
$$\mathbf{m}_1=(0,4,7,9)^T,\qquad\mathbf{m}_2=(5,9,12,14)^T.$$

Both give

$D\mathbf{m}_1=D\mathbf{m}_2=(4,3,2)^T$. Their difference is $(5,5,5,5)^T$, which is mapped to zero. Therefore, the intervals alone do not determine the absolute starting height.

In [3]:
import numpy as np

melody = np.array([0, 4, 7, 9], dtype = int)
D = np.array([
    [-1, 1, 0, 0],
    [0, -1, 1, 0],
    [0, 0, -1, 1]
], dtype = int)

intervals_matrix = D @ melody
intervals_numpy = np.diff(melody)

print("Melody: ", melody)
print("Intervals from matrix: ", intervals_matrix)
print("Intervals from np.diff: ", intervals_numpy)
assert np.array_equal(intervals_matrix, intervals_numpy)
assert np.array_equal(intervals_matrix, [4, 3, 2])

Melody:  [0 4 7 9]
Intervals from matrix:  [4 3 2]
Intervals from np.diff:  [4 3 2]


In [4]:
def reconstruct_melody(start_height: int, intervals) -> np.ndarray:
  """
  Reconstruct integer note height from the first height and directed intervals.
  """
  intervals = np.asarray(intervals, dtype = int)
  return np.concatenate(([start_height], start_height + np.cumsum(intervals)))

original = reconstruct_melody(0, [4, 3, 2])
transposed = reconstruct_melody(5, [4, 3, 2])

print("Starting at C: ", original, [note_name(x) for x in original])
print("Starting at F: ", transposed, [note_name(x) for x in transposed])

assert np.array_equal(np.diff(original), np.diff(transposed))

Starting at C:  [0 4 7 9] ['C', 'E', 'G', 'A']
Starting at F:  [ 5  9 12 14] ['F', 'A', 'C', 'D']


## 4. Chords as Semitone Patterns